# Ablations

Scores of the five compared configurations on the 945 evaluation papers (Table 8.1 of the
thesis). Each configuration uses the same reader; only its input differs.

Inputs per run folder in `experiments/`: `generations.csv`, `evaluated_result.csv` (ROUGE-L,
BERTScore) and `judge_scores.csv` (`src/evaluation/judge_scores.py`). Identification ranks each
generated abstract against the original abstracts of all 945 papers by TF-IDF cosine
(`src/adapter/identify.py`).

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "../src/adapter")
import identify

RUNS = {
    "reference mean": "reader-refs-mean",
    "GNN estimate": "reader-gnn",
    "joint model": "joint",
    "zero input": "reader-zero",
    "oracle input": "reader-oracle",
}
ASPECTS = ["subfield", "problem", "task", "coherence", "language"]

## Scores

In [2]:
rows = {}
for name, run in RUNS.items():
    d = f"../experiments/2026-10-09-14541449_run11-{run}/"
    gen = pd.read_csv(d + "generations.csv")
    sim = pd.read_csv(d + "evaluated_result.csv")
    judge = pd.read_csv(d + "judge_scores.csv")[ASPECTS].apply(pd.to_numeric, errors="coerce")
    rank = identify.ranks(gen.generated_abstract.astype(str).tolist(),
                          gen.original_abstract.astype(str).tolist())
    rows[name] = {
        "papers": len(gen),
        "distinct": gen.generated_abstract.nunique(),
        "ROUGE-L": sim.rougeL_f.mean(),
        "BERTScore": sim.bertscore_f1.mean(),
        "top-1 (%)": 100 * identify.summary(rank)["top1"],
        **judge.mean().to_dict(),
    }

table = pd.DataFrame(rows).T
table[["papers", "distinct"]] = table[["papers", "distinct"]].astype(int)
with pd.option_context("display.float_format", "{:.3f}".format):
    print(table.to_string())

                papers  distinct  ROUGE-L  BERTScore  top-1 (%)  subfield  problem  task  coherence  language
reference mean     945       938    0.175      0.844     16.508     1.102    1.247 1.299      2.850     2.542
GNN estimate       945       939    0.179      0.845     18.519     1.263    1.393 1.440      2.821     2.451
joint model        945       941    0.176      0.844     18.836     1.295    1.361 1.429      2.817     2.444
zero input         945         1    0.141      0.810      0.106     0.000    0.000 0.000      2.000     2.000
oracle input       945       945    0.184      0.848     27.831     1.576    1.583 1.616      2.752     2.407
